# Run a Qwen3.5 reasoning model

The Qwen3.5 series offers models in different sizes. Here, we use the 9B model and try to query
it in both reasoning and non-reasoning modes.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_name = "Qwen/Qwen3.5-9B"

Be sure to always load the model and the tokenizer with the same name. Otherwise, results are completely arbitrary.

In [ ]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Check memory usage

In [ ]:
!nvidia-smi

Play with different prompts and `enable_thinking`. You can also modify the system prompt!

In [ ]:
# prepare the model input
# prompt = "How many 'i's are in 'inscription'?"
prompt = "How many 'r's are in 'strawberry'?"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

The text is not yet tokenized, so you can observe the effect of the chat template.

In [ ]:
# we check what happens when we remove thinking later
text

As model inputs, we need the `id`s of the tokens.

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

Check the speed of the generation process.

In [ ]:
%%time 
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

The model returns the text including the prompt, so skip our own input.

In [ ]:
# only read output, skip input
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

Here we could calculate the number of tokens per second:

In [ ]:
len(output_ids)

Split the data into the thinking process and the solution.
You can get the `token_id`s from `tokenizer_config.json`.

And this is the solution:

In [ ]:
content = tokenizer.decode(output_ids).strip("\n")

In [ ]:
print(content)

Most models producs results in *markdown*, which can be converted to HTML:

In [ ]:
from IPython.display import display, Markdown
display(Markdown(content))

## Disable thinking

In [ ]:
# prepare the model input
# prompt = "How many 'i's are in 'inscription'?"
prompt = "How many 'r's are in 'strawberry'?"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
%%time 
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

In [ ]:
# only read output, skip input
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids).strip("\n")
display(Markdown(content))